# M2 frontier-normalized sampled LightGCN — validation smoke

Notebook này chỉ trả lời một câu: **M2 frontier-normalized có giữ quality của M1 đồng thời giảm phụ thuộc vào global hub khi mọi yếu tố ngoài proposal được giữ nguyên hay không?**

- Chỉ dùng frozen training graph và validation target.
- Mỗi BPR batch dựng `V0` từ user, positive item và negative item.
- Mỗi layer lấy đúng `min(65.536, |candidate|)` context node, không hoàn lại.
- Proposal dùng `frontier_support / sqrt(training_degree)`: ưu tiên node nối với nhiều node trong frontier hiện tại và giảm trọng số global hub.
- `K_l = V0 ∪ V_l`; không tích lũy sampled node qua layer.
- LightGCN dùng rectangular sampled-local bi-normalization, không self-loop.
- Training dùng sampled computation graph; validation luôn dùng full training graph và exact full catalog.
- M0 uniform và M1 degree-aware là matched controls; MostPop, BPR-MF và Full LightGCN chỉ là reference.
- Không đọc test, không tuning, không checkpoint, không learned policy.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không ở Colab; notebook này cần artifact tương đương và GPU CUDA.')

DRIVE_ROOT = Path('/content/drive/MyDrive/Phase2_Amazon_Audit')
GRAPH_DIR = DRIVE_ROOT / 'g2c_baby_p4'
MOSTPOP_DIR = DRIVE_ROOT / 'mostpop_validation'
BPR_DIR = DRIVE_ROOT / 'bpr_mf_sanity_v1'
FULL_LIGHTGCN_DIR = DRIVE_ROOT / 'full_lightgcn_sanity_v1'
UNIFORM_DIR = DRIVE_ROOT / 'uniform_sampling_k65536_smoke_v1'
DEGREE_DIR = DRIVE_ROOT / 'degree_aware_sampling_k65536_smoke_v1'
OUTPUT_DIR = DRIVE_ROOT / 'frontier_normalized_sampling_k65536_smoke_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = GRAPH_DIR / 'baby_p4_g2c_manifest.json'
MOSTPOP_PATH = MOSTPOP_DIR / 'mostpop_validation_summary.json'
BPR_PATH = BPR_DIR / 'bpr_mf_validation_summary.json'
FULL_LIGHTGCN_PATH = FULL_LIGHTGCN_DIR / 'full_lightgcn_validation_summary.json'
UNIFORM_PATH = UNIFORM_DIR / 'uniform_sampling_validation_summary.json'
DEGREE_PATH = DEGREE_DIR / 'degree_aware_sampling_validation_summary.json'

for required in (MANIFEST_PATH, MOSTPOP_PATH, BPR_PATH, FULL_LIGHTGCN_PATH, UNIFORM_PATH, DEGREE_PATH):
    if not required.exists():
        raise FileNotFoundError(f'Thiếu artifact: {required}')

print('Manifest:', MANIFEST_PATH)
print('MostPop:', MOSTPOP_PATH)
print('BPR-MF:', BPR_PATH)
print('Full LightGCN reference:', FULL_LIGHTGCN_PATH)
print('Uniform M0 matched control:', UNIFORM_PATH)
print('Degree-aware M1 matched control:', DEGREE_PATH)
print('Output:', OUTPUT_DIR)


In [ ]:
REGISTERED_CONFIG_FILE_SHA256 = 'b82651b29dd31a8eeaa279e2bc0c7612ee1daaa4c205f231dde39b65bef8563d'

CONFIG = {'claim_boundary': 'One fixed validation-only M2 frontier-normalized sampled-training smoke run at the predeclared k_l=65536 layer budget. This project-candidate probe tests one formula selected after reading the M0-M1 validation controls. No test access, hyperparameter search, learned policy, sampled-inference claim, or comparison to full LightGCN as a budget-matched method.', 'config_id': 'frontier-normalized-sampling-k65536-smoke-v1-2026-09-15', 'evaluation': {'candidate_universe': 'all frozen training items minus strict prior mapped positives', 'eval_batch_size': 128, 'inference_graph': 'full_frozen_training_graph', 'k': 20, 'rank_tie_break': 'item_idx_ascending', 'split': 'validation_only', 'target_unit': 'one relevant item per target row', 'timestamp_rule': 'events at the target timestamp are not prior history'}, 'matched_control_id': 'static-controls-k65536-smoke-v1', 'method_role': 'single_project_candidate_after_static_controls', 'model': {'aggregation': 'unweighted_mean_of_ego_and_all_layers', 'bias': False, 'embedding_dim': 64, 'initialization_std': 0.01, 'layers': 3, 'name': 'Sampled-LightGCN', 'recommender_self_loops': False}, 'sampler': {'candidate_rule': 'unique_neighbors_of_previous_K_excluding_previous_K', 'cross_layer_reentry': True, 'degree_exponent': -0.5, 'degree_source': 'full_frozen_training_graph_only', 'exact_k_without_replacement': True, 'frontier_support_source': 'count_training_edges_from_candidate_to_previous_K', 'k_l': [65536, 65536, 65536], 'k_l_interpretation': 'global_context_node_budget_per_batch_and_layer_equal_to_training_batch_size_for_this_smoke_only', 'method': 'frontier_normalized', 'normalization': 'rectangular_sampled_local_bi_normalization', 'positive_edge_policy': 'retain', 'priority_formula': 'log(frontier_support) - 0.5*log(training_degree) + gumbel', 'seed': 20261013, 'state_rule': 'K_l_equals_V0_union_V_l_not_cumulative_union', 'support_exponent': 1.0}, 'training': {'batch_size': 65536, 'epoch_selection': 'fixed_last_epoch', 'epochs': 5, 'l2_coefficient': 1e-06, 'learning_rate': 0.01, 'mixed_precision': False, 'negative_sampling': 'uniform_training_catalog_reject_all_training_positives', 'optimizer': 'SparseAdam', 'seed': 20260913, 'training_pair_order': 'seeded_epoch_permutation'}}

assert CONFIG['evaluation']['split'] == 'validation_only'
assert CONFIG['evaluation']['inference_graph'] == 'full_frozen_training_graph'
assert CONFIG['sampler']['method'] == 'frontier_normalized'
assert CONFIG['sampler']['degree_source'] == 'full_frozen_training_graph_only'
assert CONFIG['sampler']['frontier_support_source'] == 'count_training_edges_from_candidate_to_previous_K'
assert CONFIG['sampler']['support_exponent'] == 1.0
assert CONFIG['sampler']['degree_exponent'] == -0.5
assert CONFIG['sampler']['exact_k_without_replacement'] is True
assert CONFIG['sampler']['state_rule'] == 'K_l_equals_V0_union_V_l_not_cumulative_union'
assert CONFIG['training']['epoch_selection'] == 'fixed_last_epoch'
assert CONFIG['training']['epochs'] == 5
assert CONFIG['model']['layers'] == len(CONFIG['sampler']['k_l']) == 3
print('Frozen config:', CONFIG['config_id'])
print('Matched control:', CONFIG['matched_control_id'])
print('Registered file SHA-256:', REGISTERED_CONFIG_FILE_SHA256)


## Gate logic

1. Khóa checksum của data và ba reference summary đã review.
2. Dựng strict-prior validation history mà không đọc test.
3. Dựng training-only CSR có edge ID để audit sampled block.
4. Kiểm deterministic replay, exact-k, candidate, state và normalization.
5. Train đủ 5 epoch, mỗi training edge làm positive đúng một lần mỗi epoch.
6. Dùng full training graph cho exact validation và cohort/exposure report.
7. Bundle chỉ gồm JSON, Markdown và PNG, không có checkpoint.


In [ ]:
from collections import Counter, defaultdict
from dataclasses import dataclass
from itertools import groupby
import csv
import gc
import gzip
import hashlib
import json
import math
import platform
import resource
import time

import numpy as np
import scipy.sparse as sp

MOSTPOP_SUMMARY_SHA256 = '00332090cae73a563a7fcafde895572fb06d574366ce37bf6f9d90208dfccf62'
BPR_SUMMARY_SHA256 = '611bd820c66c57ef7a06d18e52a36a61942bfaf55fc9a615ee6b4708ac5a1f74'
FULL_LIGHTGCN_SUMMARY_SHA256 = 'ebddb3ece87abc0098106dd2271b9134f641ca8e2582e35491b9f26b6fe0faa9'
UNIFORM_SUMMARY_SHA256 = 'f29ed8f15d50a6d585ecb9cdcfc160232b64f9936e83fc8db6dac442bf941474'
DEGREE_SUMMARY_SHA256 = '46d83f7e6938c94be3376a5e71387e6a1bf9e9db32043d4dbbd10378f9bd07da'


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_artifact(entry, fallback_name):
    recorded = Path(entry['path'])
    fallback = GRAPH_DIR / fallback_name
    if recorded.exists():
        return recorded
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f'Không tìm thấy artifact: {recorded} hoặc {fallback}')


def row_metrics(ranks, k):
    ranks = np.asarray(ranks, dtype=np.int64)
    if ranks.size == 0:
        return {'rows': 0, 'hits_at_k': 0, 'recall_at_k': None, 'ndcg_at_k': None, 'k': k}
    hits = ranks <= k
    discounts = np.zeros(ranks.size, dtype=np.float64)
    discounts[hits] = 1.0 / np.log2(ranks[hits] + 1.0)
    return {
        'rows': int(ranks.size),
        'hits_at_k': int(hits.sum()),
        'recall_at_k': float(hits.mean()),
        'ndcg_at_k': float(discounts.mean()),
        'k': int(k),
    }


def item_cohort(degree):
    if degree >= 397:
        return 'head'
    if degree >= 13:
        return 'body'
    return 'tail'


def user_cohort(degree):
    if degree == 1:
        return 'singleton'
    if degree <= 3:
        return 'repeat_light'
    return 'active'


def grouped_metrics(ranks, labels, k):
    ranks = np.asarray(ranks)
    labels = np.asarray(labels)
    result = {}
    for label in sorted(set(labels.tolist())):
        mask = labels == label
        result[label] = {**row_metrics(ranks[mask], k), 'target_share': float(mask.mean())}
    return result


def process_peak_rss_mb():
    return float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0)


def positive_collision_mask(users, items, positive_keys, item_count):
    keys = users.astype(np.int64, copy=False) * item_count + items.astype(np.int64, copy=False)
    positions = np.searchsorted(positive_keys, keys)
    in_bounds = positions < positive_keys.size
    collisions = np.zeros(keys.size, dtype=bool)
    collisions[in_bounds] = positive_keys[positions[in_bounds]] == keys[in_bounds]
    return collisions


def sample_exact_uniform_negatives(users, rng, positive_keys, item_count):
    negatives = rng.integers(0, item_count, size=users.size, dtype=np.int32)
    collisions = positive_collision_mask(users, negatives, positive_keys, item_count)
    resampled = 0
    rounds = 0
    while collisions.any():
        count = int(collisions.sum())
        negatives[collisions] = rng.integers(0, item_count, size=count, dtype=np.int32)
        resampled += count
        rounds += 1
        if rounds > 100:
            raise RuntimeError('Negative rejection sampler không hội tụ')
        collisions = positive_collision_mask(users, negatives, positive_keys, item_count)
    return negatives, resampled


In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
mostpop = json.loads(MOSTPOP_PATH.read_text(encoding='utf-8'))
bpr = json.loads(BPR_PATH.read_text(encoding='utf-8'))
full_lightgcn = json.loads(FULL_LIGHTGCN_PATH.read_text(encoding='utf-8'))
uniform_m0 = json.loads(UNIFORM_PATH.read_text(encoding='utf-8'))
degree_m1 = json.loads(DEGREE_PATH.read_text(encoding='utf-8'))

uniform_config = {key: value for key, value in uniform_m0['registered_config'].items() if key != 'file_sha256'}
degree_config = {key: value for key, value in degree_m1['registered_config'].items() if key != 'file_sha256'}
matched_sampler_fields = (
    'candidate_rule', 'cross_layer_reentry', 'exact_k_without_replacement', 'k_l',
    'k_l_interpretation', 'normalization', 'positive_edge_policy', 'seed', 'state_rule',
)
matched_sampler_core_equal = all(
    uniform_config['sampler'][field] == degree_config['sampler'][field] == CONFIG['sampler'][field]
    for field in matched_sampler_fields
)
control_checks = {
    'mostpop_sha256_matches': sha256_file(MOSTPOP_PATH) == MOSTPOP_SUMMARY_SHA256,
    'bpr_sha256_matches': sha256_file(BPR_PATH) == BPR_SUMMARY_SHA256,
    'full_lightgcn_sha256_matches': sha256_file(FULL_LIGHTGCN_PATH) == FULL_LIGHTGCN_SUMMARY_SHA256,
    'uniform_sha256_matches': sha256_file(UNIFORM_PATH) == UNIFORM_SUMMARY_SHA256,
    'degree_sha256_matches': sha256_file(DEGREE_PATH) == DEGREE_SUMMARY_SHA256,
    'mostpop_integrity_passed': all(mostpop['integrity_assertions'].values()),
    'bpr_integrity_passed': all(bpr['integrity_assertions'].values()),
    'full_lightgcn_integrity_passed': all(full_lightgcn['integrity_assertions'].values()),
    'uniform_integrity_passed': all(uniform_m0['integrity_assertions'].values()),
    'degree_integrity_passed': all(degree_m1['integrity_assertions'].values()),
    'uniform_degree_frontier_matched_control_id': uniform_config['matched_control_id'] == degree_config['matched_control_id'] == CONFIG['matched_control_id'],
    'matched_model_equal': uniform_config['model'] == degree_config['model'] == CONFIG['model'],
    'matched_training_equal': uniform_config['training'] == degree_config['training'] == CONFIG['training'],
    'matched_evaluation_equal': uniform_config['evaluation'] == degree_config['evaluation'] == CONFIG['evaluation'],
    'matched_sampler_core_equal_except_proposal': matched_sampler_core_equal,
    'uniform_method_is_uniform': uniform_config['sampler']['method'] == 'uniform',
    'degree_method_is_degree_proportional': degree_config['sampler']['method'] == 'degree_proportional',
    'frontier_method_is_frontier_normalized': CONFIG['sampler']['method'] == 'frontier_normalized',
}
if not all(control_checks.values()):
    raise AssertionError(control_checks)

user_count = int(manifest['training_graph']['users'])
item_count = int(manifest['training_graph']['items'])
node_count = user_count + item_count
train_entry = manifest['artifacts']['train_edges']
validation_entry = manifest['artifacts']['validation_targets']
train_path = resolve_artifact(train_entry, 'baby_p4_train_edges.csv.gz')
validation_path = resolve_artifact(validation_entry, 'baby_p4_validation_targets.csv.gz')

print('1/7 Checksum...')
train_sha = sha256_file(train_path)
validation_sha = sha256_file(validation_path)
if train_sha != train_entry['sha256'] or validation_sha != validation_entry['sha256']:
    raise AssertionError('Checksum data không khớp manifest')

validation_rows_expected = int(validation_entry['rows'])
target_users = np.empty(validation_rows_expected, dtype=np.int32)
target_items = np.empty(validation_rows_expected, dtype=np.int32)
target_timestamps = np.empty(validation_rows_expected, dtype=np.int64)
target_source_rows = np.empty(validation_rows_expected, dtype=np.int64)
target_candidate_counts = np.empty(validation_rows_expected, dtype=np.int32)

print('2/7 Đọc validation target...')
validation_rows = 0
with gzip.open(validation_path, 'rt', encoding='utf-8', newline='') as handle:
    for index, row in enumerate(csv.DictReader(handle)):
        if index >= validation_rows_expected:
            raise AssertionError('Validation artifact có nhiều row hơn manifest')
        target_users[index] = int(row['user_idx'])
        target_items[index] = int(row['item_idx'])
        target_timestamps[index] = int(row['timestamp_ms'])
        target_source_rows[index] = int(row['source_row'])
        target_candidate_counts[index] = int(row['candidate_count'])
        validation_rows = index + 1
if validation_rows != validation_rows_expected:
    raise AssertionError('Validation row count không khớp manifest')

eval_user_mask = np.zeros(user_count, dtype=bool)
eval_user_mask[np.unique(target_users)] = True
base_history = defaultdict(list)
train_rows_expected = int(train_entry['rows'])
train_users = np.empty(train_rows_expected, dtype=np.int32)
train_items = np.empty(train_rows_expected, dtype=np.int32)

print('3/7 Đọc training edge...')
train_rows = 0
with gzip.open(train_path, 'rt', encoding='utf-8', newline='') as handle:
    for index, row in enumerate(csv.DictReader(handle)):
        if index >= train_rows_expected:
            raise AssertionError('Training artifact có nhiều row hơn manifest')
        user_idx = int(row['user_idx'])
        item_idx = int(row['item_idx'])
        train_users[index] = user_idx
        train_items[index] = item_idx
        if eval_user_mask[user_idx]:
            base_history[user_idx].append(item_idx)
        train_rows = index + 1
if train_rows != train_rows_expected:
    raise AssertionError('Training row count không khớp manifest')

print('4/7 Degree, positive keys và strict prior history...')
user_degrees = np.bincount(train_users, minlength=user_count).astype(np.int64)
item_degrees = np.bincount(train_items, minlength=item_count).astype(np.int64)
node_degrees = np.concatenate((user_degrees, item_degrees))
if int(user_degrees.sum()) != train_rows or int(item_degrees.sum()) != train_rows:
    raise AssertionError('Degree mass không bằng training rows')

positive_keys = train_users.astype(np.int64) * item_count + train_items.astype(np.int64)
positive_keys.sort()
if np.any(np.diff(positive_keys) == 0):
    raise AssertionError('Training user-item pair không unique')

targets_by_user = defaultdict(list)
for target_index, user_idx in enumerate(target_users):
    targets_by_user[int(user_idx)].append(target_index)
histories = [None] * validation_rows
target_not_in_prior = True
candidate_checks = 0
for user_idx, indices in targets_by_user.items():
    prior = set(base_history[user_idx])
    indices.sort(key=lambda index: (int(target_timestamps[index]), int(target_source_rows[index])))
    for _, timestamp_group in groupby(indices, key=lambda index: int(target_timestamps[index])):
        group = list(timestamp_group)
        for target_index in group:
            target_item = int(target_items[target_index])
            if target_item in prior:
                target_not_in_prior = False
                raise AssertionError('Validation target đã nằm trong strict prior history')
            if item_count - len(prior) != int(target_candidate_counts[target_index]):
                raise AssertionError('Candidate count không khớp strict prior history')
            histories[target_index] = np.asarray(sorted(prior), dtype=np.int32)
            candidate_checks += 1
        prior.update(int(target_items[index]) for index in group)
if any(history is None for history in histories):
    raise AssertionError('Có validation target thiếu history')

print('5/7 Negative sampler preflight...')
preflight_rng = np.random.default_rng(CONFIG['training']['seed'])
preflight_indices = preflight_rng.choice(train_rows, size=min(50000, train_rows), replace=False)
preflight_users = train_users[preflight_indices]
preflight_negatives, preflight_resamples = sample_exact_uniform_negatives(
    preflight_users, preflight_rng, positive_keys, item_count
)
if positive_collision_mask(preflight_users, preflight_negatives, positive_keys, item_count).any():
    raise AssertionError('Negative preflight có positive collision')

print('6/7 Dựng training-only symmetric CSR có edge ID...')
edge_ids = np.arange(train_rows, dtype=np.int64) + 1
global_items = user_count + train_items.astype(np.int64)
csr_rows = np.concatenate((train_users.astype(np.int64), global_items))
csr_cols = np.concatenate((global_items, train_users.astype(np.int64)))
csr_data = np.concatenate((edge_ids, edge_ids))
graph_csr = sp.csr_matrix((csr_data, (csr_rows, csr_cols)), shape=(node_count, node_count))
graph_csr.sort_indices()
if graph_csr.nnz != 2 * train_rows:
    raise AssertionError('CSR không có đúng hai direction cho mỗi training edge')

source_assertions = {
    **control_checks,
    'train_sha256_matches_manifest': train_sha == train_entry['sha256'],
    'validation_sha256_matches_manifest': validation_sha == validation_entry['sha256'],
    'train_rows_match_manifest': train_rows == int(train_entry['rows']),
    'validation_rows_match_manifest': validation_rows == int(validation_entry['rows']),
    'degree_mass_matches_train_rows': int(user_degrees.sum()) == train_rows == int(item_degrees.sum()),
    'training_pairs_unique': not np.any(np.diff(positive_keys) == 0),
    'candidate_count_matches_every_target': candidate_checks == validation_rows,
    'target_not_in_strict_prior_history': target_not_in_prior,
    'negative_sampler_preflight_has_no_positive_collision': True,
    'training_csr_has_two_directions_per_edge': graph_csr.nnz == 2 * train_rows,
    'test_targets_not_read': True,
}
if not all(source_assertions.values()):
    raise AssertionError(source_assertions)
print('7/7 Source gate pass.')


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError('Frontier-normalized sampling smoke yêu cầu Colab GPU CUDA.')
device = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = False


@dataclass
class NumpyBlock:
    source_nodes: np.ndarray
    target_nodes: np.ndarray
    source_local: np.ndarray
    target_local: np.ndarray
    weights: np.ndarray
    edge_ids: np.ndarray


def candidate_nodes_with_support(previous_nodes):
    rows = graph_csr[previous_nodes]
    neighbors, support = np.unique(
        rows.indices.astype(np.int64, copy=False), return_counts=True
    )
    positions = np.searchsorted(previous_nodes, neighbors)
    in_previous = np.zeros(neighbors.size, dtype=bool)
    bounded = positions < previous_nodes.size
    in_previous[bounded] = previous_nodes[positions[bounded]] == neighbors[bounded]
    keep = ~in_previous
    return neighbors[keep], support[keep].astype(np.int64, copy=False)


def frontier_normalized_weights(candidates, support, degrees, support_exponent, degree_exponent):
    candidates = np.asarray(candidates, dtype=np.int64)
    support = np.asarray(support, dtype=np.float64)
    if candidates.size != support.size:
        raise ValueError('Candidate và frontier support phải cùng kích thước')
    raw_degrees = np.asarray(degrees[candidates], dtype=np.float64)
    weights = np.power(support, float(support_exponent)) * np.power(raw_degrees, float(degree_exponent))
    if not np.isfinite(weights).all() or np.any(weights <= 0):
        raise ValueError('Frontier-normalized proposal yêu cầu support và degree tạo weight hữu hạn, dương')
    return weights


def frontier_normalized_exact_k(candidates, support, requested_k, rng, degrees, support_exponent, degree_exponent):
    effective_k = min(int(requested_k), int(candidates.size))
    if effective_k == 0:
        return np.empty(0, dtype=np.int64)
    weights = frontier_normalized_weights(
        candidates, support, degrees, support_exponent, degree_exponent
    )
    uniforms = np.clip(rng.random(candidates.size), np.finfo(np.float64).tiny, 1.0 - np.finfo(np.float64).eps)
    gumbels = -np.log(-np.log(uniforms))
    priorities = np.log(weights) + gumbels
    if effective_k == candidates.size:
        selected = candidates.copy()
    else:
        threshold = candidates.size - effective_k
        selected = candidates[np.argpartition(priorities, threshold)[threshold:]]
    selected.sort()
    return selected


def build_block(source_nodes, target_nodes):
    sliced = graph_csr[target_nodes]
    source_global = sliced.indices.astype(np.int64, copy=False)
    target_local_all = np.repeat(np.arange(target_nodes.size, dtype=np.int64), np.diff(sliced.indptr))
    positions = np.searchsorted(source_nodes, source_global)
    keep = positions < source_nodes.size
    valid_positions = positions[keep]
    valid_globals = source_global[keep]
    matched = source_nodes[valid_positions] == valid_globals
    keep_indices = np.flatnonzero(keep)[matched]
    source_local = positions[keep_indices].astype(np.int64, copy=False)
    target_local = target_local_all[keep_indices].astype(np.int64, copy=False)
    underlying_edge_ids = sliced.data[keep_indices].astype(np.int64, copy=False) - 1
    if source_local.size == 0:
        weights = np.empty(0, dtype=np.float32)
    else:
        source_degree = np.bincount(source_local, minlength=source_nodes.size)
        target_degree = np.bincount(target_local, minlength=target_nodes.size)
        weights = (1.0 / np.sqrt(source_degree[source_local] * target_degree[target_local])).astype(np.float32)
    return NumpyBlock(source_nodes, target_nodes, source_local, target_local, weights, underlying_edge_ids)


def build_trace(v0, rng):
    k_sets = [v0]
    blocks = []
    layers = []
    previous = v0
    for layer_index, requested_k in enumerate(CONFIG['sampler']['k_l'], start=1):
        candidates, candidate_support = candidate_nodes_with_support(previous)
        candidate_weights = frontier_normalized_weights(
            candidates, candidate_support, node_degrees,
            CONFIG['sampler']['support_exponent'], CONFIG['sampler']['degree_exponent'],
        )
        selected = frontier_normalized_exact_k(
            candidates, candidate_support, requested_k, rng, node_degrees,
            CONFIG['sampler']['support_exponent'], CONFIG['sampler']['degree_exponent'],
        )
        selected_positions = np.searchsorted(candidates, selected)
        current = np.union1d(v0, selected)
        block = build_block(current, previous)
        layers.append({
            'layer': layer_index,
            'candidate_nodes': candidates,
            'candidate_support': candidate_support,
            'candidate_weights': candidate_weights,
            'selected_nodes': selected,
            'selected_support': candidate_support[selected_positions],
            'requested_k': int(requested_k),
        })
        k_sets.append(current)
        blocks.append(block)
        previous = current
    return {'v0': v0, 'k_sets': k_sets, 'blocks': blocks, 'layers': layers}


def trace_fingerprint(trace):
    digest = hashlib.sha256()
    for layer, block in zip(trace['layers'], trace['blocks']):
        for array in (layer['candidate_nodes'], layer['candidate_support'], layer['selected_nodes'], block.edge_ids):
            digest.update(np.asarray(array, dtype=np.int64).tobytes())
    return digest.hexdigest()


def trace_contract(trace):
    exact_k = True
    unique = True
    disjoint = True
    state_rule = True
    original_edges = True
    no_self_loop = True
    finite_weights = True
    proposal_weights_positive = True
    frontier_support_positive = True
    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks']), start=1):
        candidates = layer['candidate_nodes']
        selected = layer['selected_nodes']
        previous = trace['k_sets'][index - 1]
        expected_current = np.union1d(trace['v0'], selected)
        exact_k &= selected.size == min(layer['requested_k'], candidates.size)
        unique &= np.unique(selected).size == selected.size
        disjoint &= np.intersect1d(selected, previous).size == 0
        state_rule &= np.array_equal(trace['k_sets'][index], expected_current)
        original_edges &= bool(np.all((block.edge_ids >= 0) & (block.edge_ids < train_rows)))
        if block.source_local.size:
            no_self_loop &= not np.any(block.source_nodes[block.source_local] == block.target_nodes[block.target_local])
        finite_weights &= bool(np.isfinite(block.weights).all())
        candidate_weights = frontier_normalized_weights(
            candidates, layer['candidate_support'], node_degrees,
            CONFIG['sampler']['support_exponent'], CONFIG['sampler']['degree_exponent'],
        )
        proposal_weights_positive &= bool(np.isfinite(candidate_weights).all() and np.all(candidate_weights > 0))
        frontier_support_positive &= bool(
            layer['candidate_support'].size == candidates.size
            and np.all(layer['candidate_support'] > 0)
        )
    return {
        'exact_k_every_layer': bool(exact_k),
        'sampled_nodes_unique': bool(unique),
        'sampled_nodes_disjoint_from_previous_K': bool(disjoint),
        'K_l_equals_V0_union_V_l': bool(state_rule),
        'sampled_blocks_use_only_registered_training_edges': bool(original_edges),
        'sampled_blocks_have_no_self_loop': bool(no_self_loop),
        'sampled_block_weights_finite': bool(finite_weights),
        'frontier_proposal_weights_positive': bool(proposal_weights_positive),
        'frontier_support_positive': bool(frontier_support_positive),
    }


def block_to_torch(block):
    indices = torch.from_numpy(np.stack((block.target_local, block.source_local))).to(device=device, dtype=torch.long)
    values = torch.from_numpy(block.weights).to(device=device)
    return torch.sparse_coo_tensor(
        indices, values,
        size=(block.target_nodes.size, block.source_nodes.size),
        device=device,
        is_coalesced=False,
    ).coalesce()


def verify_rectangular_direction_oracle():
    indices = torch.tensor([[0, 1], [0, 1]], device=device)
    values = torch.tensor([1.0, 1.0], device=device)
    block = torch.sparse_coo_tensor(indices, values, (2, 2), device=device).coalesce()
    source = torch.tensor([[3.0], [7.0]], device=device)
    torch.testing.assert_close(torch.sparse.mm(block, source), source)


verify_rectangular_direction_oracle()


class SampledLightGCN(nn.Module):
    def __init__(self, nodes, embedding_dim, layers, init_std):
        super().__init__()
        self.embedding = nn.Embedding(nodes, embedding_dim, sparse=True)
        self.layers = int(layers)
        nn.init.normal_(self.embedding.weight, std=init_std)

    def sampled_propagate(self, trace):
        torch_blocks = [block_to_torch(block) for block in trace['blocks']]
        depth_outputs = [self.embedding(torch.from_numpy(trace['v0']).to(device=device, dtype=torch.long))]
        for depth in range(1, self.layers + 1):
            current_nodes = trace['k_sets'][depth]
            hidden = self.embedding(torch.from_numpy(current_nodes).to(device=device, dtype=torch.long))
            for layer_index in range(depth - 1, -1, -1):
                hidden = torch.sparse.mm(torch_blocks[layer_index], hidden)
            depth_outputs.append(hidden)
        return torch.stack(depth_outputs, dim=0).mean(dim=0)

    def full_propagate(self, normalized_adjacency):
        current = self.embedding.weight
        combined = current / (self.layers + 1)
        for _ in range(self.layers):
            current = torch.sparse.mm(normalized_adjacency, current)
            combined = combined + current / (self.layers + 1)
        return combined


def endpoint_locations(v0, users, positives, negatives):
    global_users = users.astype(np.int64, copy=False)
    global_positives = user_count + positives.astype(np.int64, copy=False)
    global_negatives = user_count + negatives.astype(np.int64, copy=False)
    locations = [np.searchsorted(v0, values) for values in (global_users, global_positives, global_negatives)]
    for values, found in zip((global_users, global_positives, global_negatives), locations):
        if not np.array_equal(v0[found], values):
            raise AssertionError('Triplet endpoint không nằm trong V0')
    return locations


def update_trace_accounting(trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots):
    computation_seen[trace['v0']] = True
    for index, (layer, block) in enumerate(zip(trace['layers'], trace['blocks'])):
        selected = layer['selected_nodes']
        selected_seen[selected] = True
        computation_seen[trace['k_sets'][index + 1]] = True
        edge_seen[block.edge_ids] = True
        layer_totals[index]['candidate_node_slots'] += int(layer['candidate_nodes'].size)
        layer_totals[index]['selected_context_node_slots'] += int(selected.size)
        layer_totals[index]['directed_block_entries'] += int(block.edge_ids.size)
        selected_positions = np.searchsorted(layer['candidate_nodes'], selected)
        layer_totals[index]['candidate_support_mass'] += int(layer['candidate_support'].sum())
        layer_totals[index]['selected_support_mass'] += int(layer['selected_support'].sum())
        layer_totals[index]['candidate_degree_mass'] += int(node_degrees[layer['candidate_nodes']].sum())
        layer_totals[index]['selected_degree_mass'] += int(node_degrees[selected].sum())
        layer_totals[index]['candidate_weight_mass'] += float(layer['candidate_weights'].sum())
        layer_totals[index]['selected_weight_mass'] += float(layer['candidate_weights'][selected_positions].sum())
        selected_items = selected[selected >= user_count] - user_count
        if selected_items.size:
            degrees = item_degrees[selected_items]
            selected_item_slots['head'] += int((degrees >= 397).sum())
            selected_item_slots['body'] += int(((degrees >= 13) & (degrees <= 396)).sum())
            selected_item_slots['tail'] += int((degrees <= 12).sum())


torch.manual_seed(CONFIG['training']['seed'])
torch.cuda.manual_seed_all(CONFIG['training']['seed'])
model = SampledLightGCN(
    node_count,
    CONFIG['model']['embedding_dim'],
    CONFIG['model']['layers'],
    CONFIG['model']['initialization_std'],
).to(device)
initial_probe = model.embedding.weight[:1024].detach().cpu().clone()
optimizer = torch.optim.SparseAdam(model.parameters(), lr=CONFIG['training']['learning_rate'])

torch.cuda.reset_peak_memory_stats(device)
training_started = time.perf_counter()
epoch_records = []
optimizer_steps = 0
total_negative_resamples = 0
deterministic_replay_passed = False
all_trace_contracts = []

for epoch in range(CONFIG['training']['epochs']):
    epoch_started = time.perf_counter()
    pair_rng = np.random.default_rng(CONFIG['training']['seed'] + epoch)
    order = pair_rng.permutation(train_rows)
    negatives_all, resampled = sample_exact_uniform_negatives(train_users, pair_rng, positive_keys, item_count)
    total_negative_resamples += resampled
    sampler_rng = np.random.default_rng(CONFIG['sampler']['seed'] + epoch)

    selected_seen = np.zeros(node_count, dtype=bool)
    computation_seen = np.zeros(node_count, dtype=bool)
    edge_seen = np.zeros(train_rows, dtype=bool)
    layer_totals = [
        {'layer': layer + 1, 'requested_k_per_batch': int(CONFIG['sampler']['k_l'][layer]),
         'candidate_node_slots': 0, 'selected_context_node_slots': 0, 'directed_block_entries': 0,
         'candidate_support_mass': 0, 'selected_support_mass': 0,
         'candidate_degree_mass': 0, 'selected_degree_mass': 0,
         'candidate_weight_mass': 0.0, 'selected_weight_mass': 0.0}
        for layer in range(CONFIG['model']['layers'])
    ]
    selected_item_slots = Counter()
    sampler_seconds = 0.0
    propagation_seconds = 0.0
    examples = 0
    batches = 0
    ranking_loss_sum = 0.0
    regularization_sum = 0.0

    model.train()
    for batch_number, start in enumerate(range(0, train_rows, CONFIG['training']['batch_size'])):
        stop = min(start + CONFIG['training']['batch_size'], train_rows)
        batch_indices = order[start:stop]
        users_np = train_users[batch_indices]
        positives_np = train_items[batch_indices]
        negatives_np = negatives_all[batch_indices]
        v0 = np.unique(np.concatenate((
            users_np.astype(np.int64),
            user_count + positives_np.astype(np.int64),
            user_count + negatives_np.astype(np.int64),
        )))

        sampler_started = time.perf_counter()
        trace = build_trace(v0, sampler_rng)
        sampler_seconds += time.perf_counter() - sampler_started
        contract = trace_contract(trace)
        all_trace_contracts.append(contract)
        if not all(contract.values()):
            raise AssertionError(contract)

        if epoch == 0 and batch_number == 0:
            replay_rng = np.random.default_rng(CONFIG['sampler']['seed'])
            replay = build_trace(v0, replay_rng)
            deterministic_replay_passed = trace_fingerprint(trace) == trace_fingerprint(replay)
            if not deterministic_replay_passed:
                raise AssertionError('First-batch deterministic sampler replay fail')

        update_trace_accounting(
            trace, selected_seen, computation_seen, edge_seen, layer_totals, selected_item_slots
        )
        user_loc, positive_loc, negative_loc = endpoint_locations(v0, users_np, positives_np, negatives_np)

        torch.cuda.synchronize(device)
        propagation_started = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        propagated_v0 = model.sampled_propagate(trace)
        user_loc_t = torch.from_numpy(user_loc).to(device=device, dtype=torch.long)
        positive_loc_t = torch.from_numpy(positive_loc).to(device=device, dtype=torch.long)
        negative_loc_t = torch.from_numpy(negative_loc).to(device=device, dtype=torch.long)
        user_vec = propagated_v0[user_loc_t]
        positive_vec = propagated_v0[positive_loc_t]
        negative_vec = propagated_v0[negative_loc_t]
        ranking_loss = -F.logsigmoid(
            (user_vec * positive_vec).sum(dim=1) - (user_vec * negative_vec).sum(dim=1)
        ).mean()

        endpoint_global = np.concatenate((
            users_np.astype(np.int64),
            user_count + positives_np.astype(np.int64),
            user_count + negatives_np.astype(np.int64),
        ))
        ego = model.embedding(torch.from_numpy(endpoint_global).to(device=device, dtype=torch.long))
        batch_size_actual = stop - start
        ego = ego.reshape(3, batch_size_actual, -1)
        regularization = ego.square().sum(dim=2).sum(dim=0).mean()
        loss = ranking_loss + CONFIG['training']['l2_coefficient'] * regularization
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Loss không hữu hạn ở epoch {epoch + 1}, batch {batch_number + 1}')
        loss.backward()
        if model.embedding.weight.grad is None or not model.embedding.weight.grad.is_sparse:
            raise AssertionError('Sampled embedding gradient phải sparse')
        if not torch.isfinite(model.embedding.weight.grad._values()).all():
            raise FloatingPointError('Sparse embedding gradient không hữu hạn')
        optimizer.step()
        optimizer_steps += 1
        torch.cuda.synchronize(device)
        propagation_seconds += time.perf_counter() - propagation_started

        ranking_loss_sum += float(ranking_loss.detach().cpu()) * batch_size_actual
        regularization_sum += float(regularization.detach().cpu()) * batch_size_actual
        examples += batch_size_actual
        batches += 1
        if batch_number % 10 == 0:
            print(f'Epoch {epoch + 1}: batch {batch_number + 1}, examples {examples:,}/{train_rows:,}')

        del trace, propagated_v0, user_vec, positive_vec, negative_vec, ego, loss, ranking_loss, regularization
        if batch_number % 10 == 0:
            gc.collect()

    if examples != train_rows:
        raise AssertionError('Epoch không nhìn đủ training edge')
    mean_loss = (
        ranking_loss_sum / train_rows
        + CONFIG['training']['l2_coefficient'] * regularization_sum / train_rows
    )
    record = {
        'epoch': epoch + 1,
        'mean_loss': float(mean_loss),
        'examples': int(examples),
        'optimizer_steps': int(batches),
        'wall_seconds': time.perf_counter() - epoch_started,
        'sampler_seconds': sampler_seconds,
        'propagation_and_update_seconds': propagation_seconds,
        'throughput_examples_per_second': examples / (time.perf_counter() - epoch_started),
        'unique_selected_context_nodes': int(selected_seen.sum()),
        'unique_computation_nodes': int(computation_seen.sum()),
        'unique_training_edges_in_blocks': int(edge_seen.sum()),
        'layer_totals': layer_totals,
        'selected_item_context_slots_by_cohort': dict(selected_item_slots),
    }
    epoch_records.append(record)
    print(json.dumps(record, ensure_ascii=False, indent=2))
    del order, negatives_all, selected_seen, computation_seen, edge_seen
    gc.collect()
    torch.cuda.empty_cache()

training_wall_seconds = time.perf_counter() - training_started
training_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
parameters_changed = not torch.equal(initial_probe, model.embedding.weight[:1024].detach().cpu())
expected_batches = math.ceil(train_rows / CONFIG['training']['batch_size'])
trace_assertions = {
    key: all(contract[key] for contract in all_trace_contracts)
    for key in all_trace_contracts[0]
}
training_assertions = {
    'cuda_used': device.type == 'cuda',
    'all_registered_epochs_completed': len(epoch_records) == CONFIG['training']['epochs'],
    'each_epoch_saw_all_training_edges_as_positive_pairs': all(r['examples'] == train_rows for r in epoch_records),
    'optimizer_steps_match_batches': optimizer_steps == expected_batches * CONFIG['training']['epochs'],
    'losses_finite': all(np.isfinite(r['mean_loss']) for r in epoch_records),
    'embedding_parameters_changed': parameters_changed,
    'negative_sampler_policy_enforced': True,
    'fixed_last_epoch_used': True,
    'deterministic_first_batch_sampler_replay': deterministic_replay_passed,
    'sampled_training_used': True,
    'positive_edge_retain_policy_used': CONFIG['sampler']['positive_edge_policy'] == 'retain',
    'rectangular_sampled_local_normalization_used': CONFIG['sampler']['normalization'] == 'rectangular_sampled_local_bi_normalization',
    **trace_assertions,
}
if not all(training_assertions.values()):
    raise AssertionError(training_assertions)

optimizer.zero_grad(set_to_none=True)
del optimizer, all_trace_contracts
gc.collect()
torch.cuda.empty_cache()


In [ ]:
print('Dựng full normalized training adjacency cho validation inference...')
global_users = train_users.astype(np.int64)
global_items = user_count + train_items.astype(np.int64)
full_weights = (1.0 / np.sqrt(user_degrees[train_users] * item_degrees[train_items])).astype(np.float32)
full_src = np.concatenate((global_users, global_items))
full_dst = np.concatenate((global_items, global_users))
full_values = np.concatenate((full_weights, full_weights))
full_indices = np.stack((full_dst, full_src), axis=0)
full_adjacency = torch.sparse_coo_tensor(
    torch.from_numpy(full_indices).to(device=device, dtype=torch.long),
    torch.from_numpy(full_values).to(device=device),
    size=(node_count, node_count),
    device=device,
).coalesce()
if full_adjacency._nnz() != 2 * train_rows or not torch.isfinite(full_adjacency.values()).all():
    raise AssertionError('Full inference adjacency không hợp lệ')
del global_users, global_items, full_weights, full_src, full_dst, full_values, full_indices
gc.collect()


def resolve_boundary_ties(scores, top_values, top_indices, k):
    cutoff = top_values[:, -1]
    total_at_cutoff = (scores == cutoff[:, None]).sum(dim=1)
    selected_at_cutoff = (top_values == cutoff[:, None]).sum(dim=1)
    boundary_rows = torch.nonzero(total_at_cutoff != selected_at_cutoff, as_tuple=False).flatten()
    if boundary_rows.numel() == 0:
        return top_indices, 0
    resolved = top_indices.clone()
    for row in boundary_rows.tolist():
        row_scores = scores[row]
        row_cutoff = cutoff[row]
        better = torch.nonzero(row_scores > row_cutoff, as_tuple=False).flatten()
        tied = torch.nonzero(row_scores == row_cutoff, as_tuple=False).flatten()
        chosen = torch.cat((better, tied[:k - int(better.numel())]))
        if chosen.numel() != k:
            raise AssertionError('Không resolve được top-k boundary tie')
        resolved[row] = chosen
    return resolved, int(boundary_rows.numel())


model.eval()
k = CONFIG['evaluation']['k']
eval_batch_size = CONFIG['evaluation']['eval_batch_size']
ranks = np.empty(validation_rows, dtype=np.int32)
recommended_mask = np.zeros(item_count, dtype=bool)
exposure_counts = Counter()
boundary_tie_rows = 0
item_ids = torch.arange(item_count, device=device, dtype=torch.long)

torch.cuda.reset_peak_memory_stats(device)
evaluation_started = time.perf_counter()
with torch.no_grad():
    final_embeddings = model.full_propagate(full_adjacency)
    item_matrix = final_embeddings[user_count:]
    for start in range(0, validation_rows, eval_batch_size):
        stop = min(start + eval_batch_size, validation_rows)
        users = torch.from_numpy(target_users[start:stop]).to(device=device, dtype=torch.long)
        targets = torch.from_numpy(target_items[start:stop]).to(device=device, dtype=torch.long)
        scores = final_embeddings[users] @ item_matrix.T

        mask_rows = []
        mask_items = []
        for local_row, history in enumerate(histories[start:stop]):
            if history.size:
                mask_rows.append(np.full(history.size, local_row, dtype=np.int64))
                mask_items.append(history.astype(np.int64, copy=False))
        if mask_rows:
            scores[
                torch.from_numpy(np.concatenate(mask_rows)).to(device),
                torch.from_numpy(np.concatenate(mask_items)).to(device),
            ] = -torch.inf

        row_ids = torch.arange(stop - start, device=device)
        target_scores = scores[row_ids, targets]
        if not torch.isfinite(target_scores).all():
            raise AssertionError('Target score bị mask hoặc không hữu hạn')
        strictly_better = (scores > target_scores[:, None]).sum(dim=1)
        equal_and_lower_id = ((scores == target_scores[:, None]) & (item_ids[None, :] < targets[:, None])).sum(dim=1)
        ranks[start:stop] = (1 + strictly_better + equal_and_lower_id).cpu().numpy().astype(np.int32)

        top_values, top_indices = torch.topk(scores, k=k, dim=1, largest=True, sorted=True)
        top_indices, resolved_rows = resolve_boundary_ties(scores, top_values, top_indices, k)
        boundary_tie_rows += resolved_rows
        top_numpy = top_indices.cpu().numpy()
        recommended_mask[top_numpy.reshape(-1)] = True
        top_degrees = item_degrees[top_numpy.reshape(-1)]
        exposure_counts['head'] += int((top_degrees >= 397).sum())
        exposure_counts['body'] += int(((top_degrees >= 13) & (top_degrees <= 396)).sum())
        exposure_counts['tail'] += int((top_degrees <= 12).sum())
        if (start // eval_batch_size) % 100 == 0:
            print(f'Evaluated {stop:,}/{validation_rows:,} target')

torch.cuda.synchronize(device)
evaluation_wall_seconds = time.perf_counter() - evaluation_started
evaluation_peak_gpu_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
overall_metrics = row_metrics(ranks, k)
item_labels = np.asarray([item_cohort(item_degrees[item]) for item in target_items])
user_labels = np.asarray([user_cohort(user_degrees[user]) for user in target_users])
item_cohort_metrics = grouped_metrics(ranks, item_labels, k)
user_cohort_metrics = grouped_metrics(ranks, user_labels, k)
recommendation_slots = validation_rows * k
coverage = float(recommended_mask.mean())

mostpop_metrics = mostpop['metrics']
bpr_metrics = bpr['validation']['metrics']
full_metrics = full_lightgcn['validation']['metrics']
uniform_metrics = uniform_m0['validation']['metrics']
degree_metrics = degree_m1['validation']['metrics']
evaluation_assertions = {
    'full_graph_inference_used': True,
    'full_inference_adjacency_has_two_directions_per_edge': full_adjacency._nnz() == 2 * train_rows,
    'every_validation_target_ranked': ranks.size == validation_rows,
    'all_ranks_within_candidate_count': bool(np.all(ranks >= 1) and np.all(ranks <= target_candidate_counts)),
    'recommendation_slots_reconcile': sum(exposure_counts.values()) == recommendation_slots,
    'catalog_coverage_reconciles': abs(coverage - int(recommended_mask.sum()) / item_count) < 1e-15,
    'same_validation_rows_as_all_references': validation_rows == int(mostpop_metrics['rows']) == int(bpr_metrics['rows']) == int(full_metrics['rows']) == int(uniform_metrics['rows']) == int(degree_metrics['rows']),
    'same_gpu_as_uniform_control': torch.cuda.get_device_name(device) == uniform_m0['environment']['gpu'],
    'same_gpu_as_degree_control': torch.cuda.get_device_name(device) == degree_m1['environment']['gpu'],
    'rank_tie_break_applied': True,
    'topk_boundary_ties_resolved': True,
    'test_targets_not_read_during_evaluation': True,
}
if not all(evaluation_assertions.values()):
    raise AssertionError(evaluation_assertions)

summary = {
    'status': 'FRONTIER_NORMALIZED_SAMPLING_VALIDATION_SMOKE_EXECUTED',
    'registered_config': {**CONFIG, 'file_sha256': REGISTERED_CONFIG_FILE_SHA256},
    'source': {
        'manifest_path': str(MANIFEST_PATH),
        'train_edges': {'path': str(train_path), 'rows': train_rows, 'sha256': train_sha},
        'validation_targets': {'path': str(validation_path), 'rows': validation_rows, 'sha256': validation_sha},
        'mostpop_summary': {'path': str(MOSTPOP_PATH), 'sha256': MOSTPOP_SUMMARY_SHA256},
        'bpr_mf_summary': {'path': str(BPR_PATH), 'sha256': BPR_SUMMARY_SHA256},
        'full_lightgcn_summary': {'path': str(FULL_LIGHTGCN_PATH), 'sha256': FULL_LIGHTGCN_SUMMARY_SHA256},
        'uniform_m0_summary': {'path': str(UNIFORM_PATH), 'sha256': UNIFORM_SUMMARY_SHA256},
        'degree_m1_summary': {'path': str(DEGREE_PATH), 'sha256': DEGREE_SUMMARY_SHA256},
        'users': user_count, 'items': item_count, 'nodes': node_count,
        'directed_training_csr_entries': int(graph_csr.nnz),
    },
    'integrity_assertions': {**source_assertions, **training_assertions, **evaluation_assertions},
    'training': {
        'epochs': epoch_records,
        'wall_seconds': training_wall_seconds,
        'peak_gpu_memory_mb': training_peak_gpu_mb,
        'negative_draws': train_rows * CONFIG['training']['epochs'],
        'negative_resamples': total_negative_resamples,
        'optimizer_steps': optimizer_steps,
        'deterministic_first_batch_trace_passed': deterministic_replay_passed,
    },
    'validation': {
        'metrics': overall_metrics,
        'target_item_cohorts': item_cohort_metrics,
        'user_activity_cohorts': user_cohort_metrics,
        'recommendation_exposure': {
            'recommendation_slots': recommendation_slots,
            'unique_items_at_k': int(recommended_mask.sum()),
            'catalog_coverage_at_k': coverage,
            'item_cohort_share': {label: exposure_counts[label] / recommendation_slots for label in ('head', 'body', 'tail')},
        },
        'wall_seconds': evaluation_wall_seconds,
        'peak_gpu_memory_mb': evaluation_peak_gpu_mb,
        'topk_boundary_tie_rows': boundary_tie_rows,
    },
    'matched_comparison': {
        'comparison_boundary': 'M0 uniform and M1 degree-aware are the budget-matched controls for M2. Deltas are descriptive for one fixed validation seed; no significance or test-set claim.',
        'uniform_m0': {
            'ndcg_at_20': uniform_metrics['ndcg_at_k'],
            'recall_at_20': uniform_metrics['recall_at_k'],
            'hits_at_20': uniform_metrics['hits_at_k'],
            'catalog_coverage_at_20': uniform_m0['validation']['recommendation_exposure']['catalog_coverage_at_k'],
            'training_wall_seconds': uniform_m0['training']['wall_seconds'],
            'training_peak_gpu_memory_mb': uniform_m0['training']['peak_gpu_memory_mb'],
        },
        'degree_m1': {
            'ndcg_at_20': degree_metrics['ndcg_at_k'],
            'recall_at_20': degree_metrics['recall_at_k'],
            'hits_at_20': degree_metrics['hits_at_k'],
            'catalog_coverage_at_20': degree_m1['validation']['recommendation_exposure']['catalog_coverage_at_k'],
            'training_wall_seconds': degree_m1['training']['wall_seconds'],
            'training_peak_gpu_memory_mb': degree_m1['training']['peak_gpu_memory_mb'],
        },
        'frontier_m2': {
            'ndcg_at_20': overall_metrics['ndcg_at_k'],
            'recall_at_20': overall_metrics['recall_at_k'],
            'hits_at_20': overall_metrics['hits_at_k'],
            'catalog_coverage_at_20': coverage,
            'training_wall_seconds': training_wall_seconds,
            'training_peak_gpu_memory_mb': training_peak_gpu_mb,
        },
        'frontier_minus_uniform': {
            'ndcg_at_20': overall_metrics['ndcg_at_k'] - uniform_metrics['ndcg_at_k'],
            'recall_at_20': overall_metrics['recall_at_k'] - uniform_metrics['recall_at_k'],
            'hits_at_20': overall_metrics['hits_at_k'] - uniform_metrics['hits_at_k'],
            'catalog_coverage_at_20': coverage - uniform_m0['validation']['recommendation_exposure']['catalog_coverage_at_k'],
            'training_wall_seconds': training_wall_seconds - uniform_m0['training']['wall_seconds'],
            'training_peak_gpu_memory_mb': training_peak_gpu_mb - uniform_m0['training']['peak_gpu_memory_mb'],
        },
        'frontier_minus_degree': {
            'ndcg_at_20': overall_metrics['ndcg_at_k'] - degree_metrics['ndcg_at_k'],
            'recall_at_20': overall_metrics['recall_at_k'] - degree_metrics['recall_at_k'],
            'hits_at_20': overall_metrics['hits_at_k'] - degree_metrics['hits_at_k'],
            'catalog_coverage_at_20': coverage - degree_m1['validation']['recommendation_exposure']['catalog_coverage_at_k'],
            'training_wall_seconds': training_wall_seconds - degree_m1['training']['wall_seconds'],
            'training_peak_gpu_memory_mb': training_peak_gpu_mb - degree_m1['training']['peak_gpu_memory_mb'],
        },
    },
    'reference_comparison': {
        'comparison_boundary': 'MostPop, BPR-MF and full LightGCN are context references, not budget-matched controls.',
        'mostpop': {'ndcg_at_20': mostpop_metrics['ndcg_at_k'], 'recall_at_20': mostpop_metrics['recall_at_k']},
        'bpr_mf': {'ndcg_at_20': bpr_metrics['ndcg_at_k'], 'recall_at_20': bpr_metrics['recall_at_k']},
        'full_lightgcn': {'ndcg_at_20': full_metrics['ndcg_at_k'], 'recall_at_20': full_metrics['recall_at_k']},
        'uniform_m0': {'ndcg_at_20': uniform_metrics['ndcg_at_k'], 'recall_at_20': uniform_metrics['recall_at_k']},
        'degree_m1': {'ndcg_at_20': degree_metrics['ndcg_at_k'], 'recall_at_20': degree_metrics['recall_at_k']},
        'frontier_m2': {'ndcg_at_20': overall_metrics['ndcg_at_k'], 'recall_at_20': overall_metrics['recall_at_k']},
    },
    'environment': {
        'python': platform.python_version(), 'platform': platform.platform(),
        'torch': torch.__version__, 'cuda': torch.version.cuda,
        'gpu': torch.cuda.get_device_name(device), 'process_peak_rss_mb': process_peak_rss_mb(),
    },
    'claim_boundary': CONFIG['claim_boundary'],
}
print(json.dumps({
    'status': summary['status'],
    'metrics': overall_metrics,
    'coverage_at_20': coverage,
    'training_wall_seconds': training_wall_seconds,
    'training_peak_gpu_mb': training_peak_gpu_mb,
    'all_assertions_pass': all(summary['integrity_assertions'].values()),
}, ensure_ascii=False, indent=2))


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs = [row['epoch'] for row in epoch_records]
axes[0].plot(epochs, [row['mean_loss'] for row in epoch_records], marker='o', color='#0B0D10')
axes[0].set_title('M2 frontier-normalized training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean loss')
axes[0].grid(alpha=0.25)

names = ['MostPop', 'BPR-MF', 'Full LightGCN', 'Uniform M0', 'Degree M1', 'Frontier M2']
ndcg = [mostpop_metrics['ndcg_at_k'], bpr_metrics['ndcg_at_k'], full_metrics['ndcg_at_k'], uniform_metrics['ndcg_at_k'], degree_metrics['ndcg_at_k'], overall_metrics['ndcg_at_k']]
recall = [mostpop_metrics['recall_at_k'], bpr_metrics['recall_at_k'], full_metrics['recall_at_k'], uniform_metrics['recall_at_k'], degree_metrics['recall_at_k'], overall_metrics['recall_at_k']]
x = np.arange(len(names))
width = 0.36
a = axes[1].bar(x - width / 2, ndcg, width, label='NDCG@20', color='#3D8DFF')
b = axes[1].bar(x + width / 2, recall, width, label='Recall@20', color='#2B9B75')
axes[1].set_xticks(x, names, rotation=10)
axes[1].set_title('Validation references')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].legend(frameon=False)
axes[1].grid(axis='y', alpha=0.25)

sampler_times = [row['sampler_seconds'] for row in epoch_records]
prop_times = [row['propagation_and_update_seconds'] for row in epoch_records]
axes[2].bar(epochs, sampler_times, label='Sampler', color='#F2A65A')
axes[2].bar(epochs, prop_times, bottom=sampler_times, label='Propagation + update', color='#3D8DFF')
axes[2].set_title('Training time breakdown')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Seconds')
axes[2].legend(frameon=False)
axes[2].grid(axis='y', alpha=0.25)

for axis in axes:
    axis.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
figure_path = OUTPUT_DIR / '01_frontier_normalized_sampling_validation_smoke.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
import zipfile

summary_path = OUTPUT_DIR / 'frontier_normalized_sampling_validation_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
lines = [
    '# M2 frontier-normalized sampled LightGCN — validation smoke', '',
    f"- Config: {CONFIG['config_id']}",
    f"- Matched control ID: {CONFIG['matched_control_id']}",
    f"- Layer budget: {CONFIG['sampler']['k_l']}",
    f"- NDCG@20: {overall_metrics['ndcg_at_k']:.6f}",
    f"- Recall@20: {overall_metrics['recall_at_k']:.6f}",
    f"- Catalog Coverage@20: {coverage:.6f}",
    f"- Training wall time: {training_wall_seconds:.2f} s",
    f"- Training peak GPU memory: {training_peak_gpu_mb:.2f} MB", '',
    '## Matched comparison với M0 uniform', '',
    f"- ΔNDCG@20: {overall_metrics['ndcg_at_k'] - uniform_metrics['ndcg_at_k']:+.6f}",
    f"- ΔRecall@20: {overall_metrics['recall_at_k'] - uniform_metrics['recall_at_k']:+.6f}",
    f"- ΔHit@20: {overall_metrics['hits_at_k'] - uniform_metrics['hits_at_k']:+d}",
    f"- ΔCoverage@20: {coverage - uniform_m0['validation']['recommendation_exposure']['catalog_coverage_at_k']:+.6f}", '',
    '## Matched comparison với M1 degree-aware', '',
    f"- ΔNDCG@20: {overall_metrics['ndcg_at_k'] - degree_metrics['ndcg_at_k']:+.6f}",
    f"- ΔRecall@20: {overall_metrics['recall_at_k'] - degree_metrics['recall_at_k']:+.6f}",
    f"- ΔHit@20: {overall_metrics['hits_at_k'] - degree_metrics['hits_at_k']:+d}",
    f"- ΔCoverage@20: {coverage - degree_m1['validation']['recommendation_exposure']['catalog_coverage_at_k']:+.6f}", '',
    '## Validation theo target item cohort', '',
    '| Cohort | Target share | NDCG@20 | Recall@20 |',
    '|---|---:|---:|---:|',
]
for label in ('head', 'body', 'tail'):
    row = item_cohort_metrics[label]
    lines.append(f"| {label} | {row['target_share']:.2%} | {row['ndcg_at_k']:.6f} | {row['recall_at_k']:.6f} |")
lines += ['', '## Claim boundary', '', CONFIG['claim_boundary']]
report_path = OUTPUT_DIR / 'FRONTIER_NORMALIZED_SAMPLING_VALIDATION_vn.md'
report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

bundle_path = OUTPUT_DIR / 'frontier_normalized_sampling_validation_bundle.zip'
artifacts = [summary_path, report_path, figure_path]
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact in artifacts:
        archive.write(artifact, arcname=artifact.name)
with zipfile.ZipFile(bundle_path) as archive:
    names = set(archive.namelist())
expected = {artifact.name for artifact in artifacts}
if names != expected or any(name.endswith(('.pt', '.pth', '.ckpt')) for name in names):
    raise AssertionError(f'Bundle chứa artifact ngoài contract: {sorted(names)}')
print('Summary:', summary_path)
print('Bundle:', bundle_path)
print('Bundle SHA-256:', sha256_file(bundle_path))


## Sau khi chạy

1. Xác nhận output cuối có `all_assertions_pass: true`.
2. Tải đúng `frontier_normalized_sampling_validation_bundle.zip`.
3. Không chạy test split và không tự đổi budget khi kết quả thấp hoặc runtime dài.
4. Gửi nguyên ZIP cho Codex. Chưa dùng test split và chưa mở thêm sampler khác.


In [ ]:
try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print('Bundle nằm tại:', bundle_path)
